<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🔗</div><div style="color:white;font-size:1.5rem;font-weight:700;">Collocation Analyser</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Measure how strongly words attract each other in your corpus<br><a href="https://ladal.edu.au/coll.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>

<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">How to use this tool:</b><ol style="margin:.5rem 0 0 0;padding-left:1.3em;line-height:1.9;"><li>Run <b>Step 1</b> to set up the environment.</li><li>Upload your <code>.txt</code> files to the <b>MyTexts</b> folder, then run <b>Step 2</b>.</li><li>In <b>Step 3</b>, optionally enter a node word, adjust settings, and click <b>Run Analysis</b>.</li><li>Download results from the <b>MyOutput</b> folder.</li></ol></div>

<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">&#x1F4D6; <b>This notebook contains code cells.</b> Do not edit the code — just run each step using the <b>Run Step</b> buttons. To run a cell: click on it and press <b>Shift + Enter</b>, or use the <b>&#x25B6; Run</b> button in the toolbar above.</div>

<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 1</span><br><b style="font-size:1rem;">&#x2699;&#xFE0F; Set up the environment — run this first</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;"><b>Run the cell below once</b> to load all the helper functions this notebook needs. Click on the cell, then press <b>Shift + Enter</b>. You should see a green <em>Setup complete</em> message when it finishes.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 1 — environment setup — run this cell, do not edit it</em></div>

In [ ]:
# Setup: load display helpers and shared functions
# Run this cell once — then proceed to the steps below.

suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))
suppressPackageStartupMessages(library(ipywidgets))

# ── Colour-coded feedback boxes ──────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:", bg,
    ";border-left:4px solid ", border,
    ";border-radius:6px;padding:10px 15px;margin:.4rem 0;",
    "font-family:sans-serif;font-size:.9rem;\">",
    if (nzchar(icon)) paste0(icon, " ") else "",
    msg, "</div>"))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar ─────────────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    "<div style=\"font-family:sans-serif;font-size:.85rem;margin:.3rem 0;\">",
    "<span style=\"color:#51247a;font-weight:600;\">", label, "</span>",
    "<div style=\"background:#e8e4f0;border-radius:4px;height:8px;margin-top:3px;\">",
    "<div style=\"background:#51247a;width:", pct,
    "%;height:8px;border-radius:4px;\"></div></div></div>"))
}

# ── Load plain-text files from a folder ──────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in ", folder,
         ". Please upload your files first.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts to MyOutput ───────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save data frame as Excel ─────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Setup complete. Follow the steps below.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 2</span><br><b style="font-size:1rem;">&#x1F4C2; Upload your texts to the MyTexts folder</b></div>

<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Look at the <b>file browser panel</b> on the left side of the screen.</li><li>Double-click the <b><code>MyTexts</code></b> folder to open it.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>When done, click the <b>Run Step</b> button below.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. You can upload multiple files at once.</p></div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 2 — load uploaded files — run this cell, do not edit it</em></div>

In [ ]:
# Run this cell to load your uploaded files.
btn_load <- ipywidgets::Button(
  description = "  Run Step: Load Files",
  button_style = "info", icon = "folder-open",
  layout = ipywidgets::Layout(width="220px", height="36px"))
out_load <- ipywidgets::Output()
ipywidgets::display(ipywidgets::VBox(list(btn_load, out_load)))

ipywidgets::observe(btn_load, "clicks", function(change) {
  ipywidgets::with_output(out_load, {
    tryCatch({
      texts <<- load_texts("notebooks/MyTexts")
      ok(paste0("Loaded <b>", length(texts),
                " file(s)</b>: ",
                paste(names(texts), collapse=", ")))
    }, error=function(e) err(conditionMessage(e)))
  })
})


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 3</span><br><b style="font-size:1rem;">&#x1F4CA; Configure and run the analysis</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Set your options using the controls below, then click the green <b>Run Analysis</b> button. Results will appear directly below the button and will also be saved to the <b>MyOutput</b> folder.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 3 — analysis — run this cell, do not edit it</em></div>

In [ ]:
# Configure collocation analysis and click Run Analysis.
suppressPackageStartupMessages(library(quanteda))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(writexl))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))

w_node <- ipywidgets::Text(
  value="", description="Node word:",
  placeholder="leave blank for all word pairs",
  style=list(description_width="120px"),
  layout=ipywidgets::Layout(width="380px"))
w_win  <- ipywidgets::IntSlider(
  value=5, min=1, max=15, step=1,
  description="Window size (words):",
  style=list(description_width="180px"),
  layout=ipywidgets::Layout(width="440px"))
w_minf <- ipywidgets::IntSlider(
  value=2, min=1, max=20, step=1,
  description="Min. co-occurrence count:",
  style=list(description_width="180px"),
  layout=ipywidgets::Layout(width="440px"))
w_topn <- ipywidgets::IntSlider(
  value=20, min=5, max=50, step=5,
  description="Top N in plot:",
  style=list(description_width="180px"),
  layout=ipywidgets::Layout(width="440px"))
w_btn  <- ipywidgets::Button(
  description="  Run Analysis",
  button_style="success", icon="search",
  layout=ipywidgets::Layout(width="180px", height="38px"))
w_out  <- ipywidgets::Output()

ipywidgets::display(ipywidgets::VBox(list(
  ipywidgets::HTML(paste0(
    "<div style=\"background:#f0f7ff;border:1px solid #4085C6;",
    "border-radius:6px;padding:14px 18px;margin:.5rem 0;\">",
    "<b style=\"color:#2a5ea8;\">Configure collocation analysis:</b>",
    "<br><small style=\"color:#555;\">Leave Node word blank to get ",
    "all collocations in the corpus.</small></div>")),
  w_node, w_win, w_minf, w_topn, w_btn, w_out
)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      if (!exists("texts")) stop("No files loaded. Run Step 2 first.")
      .prog("Building co-occurrence matrix...", 30)
      toks    <- tokens(texts, remove_punct=TRUE, remove_numbers=TRUE) |>
        tokens_tolower()
      fcm_obj <- fcm(toks, context="window", window=w_win$value,
                     ordered=FALSE, tri=FALSE)
      node <- tolower(trimws(w_node$value))
      if (nzchar(node)) {
        if (!node %in% featnames(fcm_obj))
          stop("Word \"", node, "\" not found. Check spelling or reduce min. frequency.")
        fcm_obj <- fcm_select(fcm_obj, pattern=node, valuetype="fixed")
      }
      .prog("Calculating MI, t-score, log-ratio...", 65)
      wf  <- colSums(dfm(toks)); N <- sum(wf)
      fcm_df <- as.data.frame(as.matrix(fcm_obj)) |>
        tibble::rownames_to_column("w1") |>
        tidyr::pivot_longer(-w1, names_to="w2", values_to="O") |>
        dplyr::filter(O >= w_minf$value, w1 != w2) |>
        dplyr::mutate(
          f1=as.integer(wf[w1]), f2=as.integer(wf[w2]),
          E=pmax((f1*f2)/N, .Machine$double.eps),
          MI=log2(O/E),
          t_score=(O-E)/sqrt(pmax(O,1)),
          log_ratio=log2((O/N)/(E/N))
        ) |>
        dplyr::select(w1,w2,O,f1,f2,MI,t_score,log_ratio) |>
        dplyr::arrange(dplyr::desc(MI))
      .prog("Plotting...", 85)
      p <- ggplot(head(fcm_df, w_topn$value),
                  aes(x=reorder(w2,MI), y=MI)) +
        geom_col(fill="#51247a", width=.7) + coord_flip() +
        theme_bw(base_size=12) +
        labs(title=paste("Top", w_topn$value,
               if(nzchar(node)) paste0("collocates of \"",node,"\"") else "collocations",
               "by MI"),
             x="Collocate", y="Mutual Information (MI)")
      print(p)
      .prog("Saving...", 95)
      save_excel(fcm_df, "collocations.xlsx")
      dir.create("notebooks/MyOutput", showWarnings=FALSE, recursive=TRUE)
      ggsave("notebooks/MyOutput/collocations_plot.png",
             bg="white", width=8, height=5, dpi=200)
      .prog("Done.", 100)
      ok("Saved <b>collocations.xlsx</b> and <b>collocations_plot.png</b> to MyOutput.")
    }, error=function(e) err(conditionMessage(e)))
  })
})


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 99</span><br><b style="font-size:1rem;">&#x2B07;&#xFE0F; Download your results</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Your results have been saved to the <b>MyOutput</b> folder. To download: open the <b>file browser panel</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> on a file and choose <b>Download</b>.</div>

---

### Citation

Schweinberger, Martin. (2025). *LADAL Collocation Analyser*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025collocations,
  author = {Schweinberger, Martin},
  title  = {LADAL Collocation Analyser},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()